In [ ]:
#Email Spam Detection Using Machine Learning(Jue Jue Khaing Win)

In [9]:
pip install gradio pandas joblib nltk scikit-learn google-auth-oauthlib google-api-python-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import gradio as gr
import pandas as pd
import joblib
import nltk
import string
import os

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]
MAX_EMAILS = 20
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

nltk.download("stopwords")
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = "".join(c for c in text if c not in string.punctuation)
    words = [stemmer.stem(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

def get_model():

    model_path = "spam_model.pkl"
    tfidf_path = "tfidf_vectorizer.pkl"

    if os.path.exists(model_path) and os.path.exists(tfidf_path):
        model = joblib.load(model_path)
        tfidf = joblib.load(tfidf_path)
        print("Model loaded successfully.")

    df = pd.read_csv("spam_ham_dataset.csv")

    df = df[['label', 'text']]
    df['label'] = df['label'].map({'ham': 0, 'spam': 1})
    
    df["text"] = df["text"].apply(clean_text)
    y = df['label']

    X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], test_size=0.2, random_state=42)
    
    tfidf = TfidfVectorizer(max_features=7000) 
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)
    
    model = MultinomialNB()
    model.fit(X_train_vec, y_train)
    
    pred = model.predict(X_test_vec)
    acc=accuracy_score(y_test, pred)
    print(f"Accuracy: {acc}")

    joblib.dump(model, model_path)
    joblib.dump(tfidf, tfidf_path)

    print("Trained and saved successfully.")
    return model, tfidf

model, tfidf = get_model()

def authenticate_gmail():
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=0)
    return creds 
    
def explain_prediction(email_text):
    if not email_text or not email_text.strip():
        return "⚠️ Please enter text", [], "No data.", {}, ""

    cleaned = clean_text(email_text) 
    vec = tfidf.transform([cleaned]) 
    
    probs = model.predict_proba(vec)[0] 
    prediction = model.predict(vec)[0] 
    res_label = "SPAM" if prediction == 1 else "NOT SPAM"
    conf_dict = {res_label: float(max(probs))}
    
    feature_names = tfidf.get_feature_names_out()
    log_prob_diff = model.feature_log_prob_[1] - model.feature_log_prob_[0]
    word_scores = dict(zip(feature_names, log_prob_diff))
    
    words = email_text.split()
    highlights = []
    relevant_keywords = set()

    for word in words:
        clean_w = "".join(c for c in word.lower() if c not in string.punctuation)
        stemmed_w = stemmer.stem(clean_w) #
        score = word_scores.get(stemmed_w, 0)

        if prediction == 1:
            if score > 0.5:
                highlights.append((word, "SPAM WORD"))
                relevant_keywords.add(word.lower())
            else:
                highlights.append((word, None))
        
        else:
            if score < -0.5:
                highlights.append((word, "HAM WORD"))
                relevant_keywords.add(word.lower())
            else:
                highlights.append((word, None))

    result_text = f"### Result: {'🚨 Detected as Spam' if prediction == 1 else '✅ Detected as Safe'}"
    
    keyword_type = "Spam Keywords" if prediction == 1 else "Safe (Ham) Keywords"
    keyword_list_str = ", ".join(list(relevant_keywords)[:10]) if relevant_keywords else "None"
    
    final_info = f"🔍 **{keyword_type} found:** {keyword_list_str}"

    return result_text, highlights, "Analysis complete.", conf_dict, final_info

def scan_gmail(creds, progress=gr.Progress()):
    try:
        if creds is None:
            return "❌ Please login first."

        service = build("gmail", "v1", credentials=creds)

        results = service.users().labels().list(userId="me").execute()
        labels = results.get("labels", [])
        target_name = "ML_Spam"

        label_id = next(
            (l["id"] for l in labels if l["name"].lower() == target_name.lower()),
            None
        )

        if not label_id:
            label_body = {
                "name": target_name,
                "labelListVisibility": "labelShow",
                "messageListVisibility": "show"
            }
            label_id = service.users().labels().create(
                userId="me",
                body=label_body
            ).execute()["id"]

        results = service.users().messages().list(
            userId="me",
            q="label:INBOX",
            maxResults=MAX_EMAILS
        ).execute()

        messages = results.get("messages", [])

        if not messages:
            return "📭 Inbox is empty."

        logs, spam_ids = [], []

        for m in messages:
            msg = service.users().messages().get(
                userId="me",
                id=m["id"],
                format="minimal"
            ).execute()

            snippet = msg.get("snippet", "")

            cleaned = clean_text(snippet)
            is_spam = model.predict(tfidf.transform([cleaned]))[0] == 1

            status = "🚨 Spam" if is_spam else "✅ Not Spam"
            logs.append(f"[{status}] {snippet[:50]}...")

            if is_spam:
                spam_ids.append(m["id"])

        if spam_ids:
            service.users().messages().batchModify(
                userId="me",
                body={
                    "ids": spam_ids,
                    "addLabelIds": [label_id],
                    "removeLabelIds": ["INBOX"]
                }
            ).execute()

            logs.append(f"\n✅ {len(spam_ids)} emails moved to '{target_name}'.")
        else:
            logs.append("\n🎉 No spam detected.")

        return "\n".join(logs)

    except Exception as e:
        return f"❌ Error: {str(e)}"

with gr.Blocks(title="Spam Detection") as app:
    gr.Markdown("# 📧 Email Spam Detection")
    
    with gr.Tab("Check Email Text"):
        with gr.Row():
            with gr.Column(scale=2):
                inp = gr.Textbox(label="Email Text Input", placeholder="Enter email text here...", lines=10)
                with gr.Row():
                    clear_btn = gr.Button("🧹 Clear Input")
                    analyze_btn = gr.Button("🔍 Analyze Email", variant="primary")
                
            with gr.Column(scale=1):
                out_conf = gr.Label(label="Confidence Score", num_top_classes=1)
                out_label = gr.Markdown()
                out_indicators = gr.Markdown()

        gr.HTML("<hr>")
        
        out_highlight = gr.HighlightedText(label="Word Visualizer (Spam vs Ham Patterns)", color_map={"SPAM WORD": "red", "HAM WORD": "green"})
        
        out_keyword_info = gr.Markdown()        
        
        analyze_btn.click(fn=explain_prediction, inputs=inp, outputs=[out_label, out_highlight, out_indicators, out_conf, out_keyword_info])
        
        clear_btn.click(fn=lambda: ("", [], "", {}, ""), outputs=[inp, out_highlight, out_label, out_conf, out_keyword_info])

    with gr.Tab("Scan Gmail Inbox"):
        gr.Markdown("### 🔐 Gmail Login Required")
        creds_state = gr.State()
        gmail_out = gr.Textbox(label="Activity Log", lines=10)
    
        login_btn = gr.Button("🔑 Login with Gmail")
        scan_btn = gr.Button("📡 Start Scanning Inbox")
    
        login_btn.click(fn=authenticate_gmail, outputs=creds_state)
        scan_btn.click(fn=scan_gmail, inputs=creds_state, outputs=gmail_out)

    with gr.Tab("About Us"):
        gr.Markdown("# 📋 Project Details")
        gr.Markdown("ဤစနစ်သည် **Machine Learning (ML)** နှင့် **Natural Language Processing (NLP)** နည်းပညာများကို ပေါင်းစပ်၍ Email များအား အလိုအလျောက် ခွဲခြားစိတ်ဖြာပေးသော System တစ်ခုဖြစ်ပါသည်။")       
        
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🤖 Our Model & Features")
                gr.Markdown("""
                * **Algorithm:** Multinomial Naive Bayes ကို အသုံးပြုထားပြီး ၎င်းသည် စာသားများကို ခွဲခြားရာတွင် အလွန်မြန်ဆန်ပြီး တိကျမှုရှိသော algorithm ဖြစ်ပါသည်။
                * **Feature Extraction:** **TF-IDF Vectorizer** (Term Frequency-Inverse Document Frequency) သုံးထားပြီး အရေးကြီးသောစကားလုံးပေါင်း (7,000 features) အထိ ခွဲခြားစိတ်ဖြာနိုင်ရန် သတ်မှတ်ထားပါသည်။
                * **Data Processing:** **PorterStemmer** နှင့် **NLTK Stopwords** တို့ကိုသုံး၍ စာသားများကို မူလရင်းမြစ်စကားလုံးများအဖြစ် ပြောင်းလဲထားပါသည်။
                * **Gmail Scan Limit:** စနစ်၏ မြန်ဆန်မှုအတွက် နောက်ဆုံးဝင်လာသော Email **အစောင် 20** ကို Scan ဖတ်ပေးပါသည်။
                * **Explainable AI:** စကားလုံးတစ်လုံးချင်းစီ၏ Spam/Not Spam score များကို Highlight ပြပေးထားသဖြင့် Model က ဘယ်လိုအချက်တွေကြောင့် ဆုံးဖြတ်လိုက်သည်ကို ပွင့်လင်းမြင်သာစွာ သိရှိနိုင်ပါသည်။
                """)

                gr.HTML("<hr>")
            
                gr.Markdown("### 📊 Dataset Information")
                gr.Markdown("""
                * **Dataset:** `spam_ham_dataset.csv` ကို အသုံးပြုထားပါသည်။
                * **Data Splitting:** Model ၏ တိကျမှုကို စမ်းသပ်ရန် Dataset ၏ **80% ကို Training** နှင့် **20% ကို Testing** အဖြစ် ခွဲဝေအသုံးပြုထားပါသည်။
                * **Labeling:** Email များကို ခွဲခြားရန်အတွက် **Ham ကို 0** နှင့် **Spam ကို 1** အဖြစ် သတ်မှတ်ထားပါသည်။
                """)


            with gr.Column():
                gr.Markdown("## 🛠️ Key Libraries & Tools")
                gr.Markdown("""
                    * **Pandas:** Dataset ကို ဖတ်ရန်၊ Data Labeling/Cleaning ပြုလုပ်ရန်အတွက် အသုံးပြုထားပါသည်။
                    * **Scikit-Learn (Sklearn):** ဤ Library ကို အသုံးပြု၍ **TF-IDF Vectorization** (စာသားမှ ကိန်းဂဏန်းပြောင်းခြင်း)၊ **Train-Test Splitting** (ဒေတာခွဲဝေခြင်း) နှင့် **Multinomial Naive Bayes** Model တည်ဆောက်ခြင်းတို့ကို လုပ်ဆောင်ပါသည်။
                    * **Joblib:** Trained model နှင့် TF-IDF vectorizer ကို save / load ပြုလုပ်ရန် အသုံးပြုထားပါသည်။
                    * **NLTK (Natural Language Toolkit):** စာသားများထဲမှ မလိုအပ်သော စကားလုံးများကို ဖယ်ရှားခြင်း (**Stopwords Removal**) နှင့် မူလရင်းမြစ်စကားလုံးများအဖြစ် ပြောင်းလဲခြင်း (**Stemming**) တို့အတွက် အသုံးပြုထားပါသည်။
                    * **Gradio:** အသုံးပြုသူများ လွယ်ကူစွာ စမ်းသပ်နိုင်ရန်အတွက် ခေတ်မီပြီး အပြန်အလှန် တုံ့ပြန်နိုင်သော **Web User Interface** ကို တည်ဆောက်ရာတွင် အသုံးပြုထားပါသည်။
                    * **Google Auth OAuthlib:** Gmail Account နှင့် လုံခြုံစွာ Authentication ပြုလုပ်ရန် အသုံးပြုထားပါသည်။
                    * **Google API Client:** Gmail Inbox နှင့် တိုက်ရိုက်ချိတ်ဆက်၍ Email များကို Scan ဖတ်ရန်နှင့် Spam များကို သီးသန့် Label အောက်သို့ အလိုအလျောက် ရွှေ့ပြောင်းရန် အသုံးပြုထားပါသည်။
                    """)

        gr.HTML("<hr>")
        
        gr.Markdown("### 🔄 System Workflow (အလုပ်လုပ်ပုံ အဆင့်ဆင့်)")
        with gr.Row():
            with gr.Column():
                gr.Markdown("""
                **Preprocessing**:
                User က input ပေးလိုက်တဲ့ email text ကို lowercase conversion, stopword removal နှင့် stemming ပြုလုပ်ပြီး clean text အဖြစ်ပြင်ဆင်ထားပါသည်။
                """)

            with gr.Column():
                gr.Markdown("""
                **Feature Extraction & Classification**:
                ပြုပြင်ပြီးသား စာသားများကို TF-IDF စနစ်ဖြင့် ကိန်းဂဏန်းပြောင်းလဲကာ Naive Bayes Model မှ Spam ဟုတ်မဟုတ်ကို စက္ကန့်ပိုင်းအတွင်း ဆုံးဖြတ်ထားပါသည်။
                """)
                
            with gr.Column():
                gr.Markdown("""
                **Model Visualization**:
                Model ၏ ရလဒ်များကို နားလည်လွယ်စေရန် Confidence Score များကို ရာခိုင်နှုန်းဖြင့် ပြသပေးထားပြီး စကားလုံးများ၏ အဓိပ္ပာယ်သက်ရောက်မှုကို အရောင်များဖြင့် ခွဲခြားပြသထားပါသည်။
                """)
           
            with gr.Column():
                gr.Markdown("""
                **Automation Scan**:
                Gmail နှင့် ချိတ်ဆက်ထားပြီး Spam ဟု ယူဆရသော Email များကို Inbox ထဲတွင် **'ML_Spam'** label သို့ အလိုအလျောက် ပြောင်းထားပါသည်။
                """)

        gr.HTML("<hr>")
       
        gr.Markdown("### 🎯 Project Goal")
        gr.Markdown("""
        အသုံးပြုသူများ၏ Gmail Inbox ထဲရှိ Spam Email များကို Machine Learning နည်းလမ်းဖြင့် အလိုအလျောက် ရှာဖွေဖယ်ရှားပေးရန်နှင့် 
        Email စာသားများကို စက္ကန့်ပိုင်းအတွင်း ခွဲခြားစိတ်ဖြာပေးနိုင်ရန် ရည်ရွယ်ပါသည်။
        """)


if __name__ == "__main__":
    app.launch(share=True)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Model loaded successfully.
Accuracy: 0.9652173913043478
Trained and saved successfully.
* Running on local URL:  http://127.0.0.1:7875
* Running on public URL: https://d8947993a55e55db10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=959481252066-un38f7sf9gol8fuk98o1kg31fc6ddtof.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A52104%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=SxU1cHrkH1R7bD25Spp3I6rmGr2VbC&access_type=offline
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=959481252066-un38f7sf9gol8fuk98o1kg31fc6ddtof.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A52122%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=3uG0F49ox3NqOWMRyrIGUkm9FGCc08&access_type=offline
